# Why the "grid" layout scores so high — and why it's a `recon` artifac

## Setup
Load the optimization module by path (its filename starts with a digit, so it can't be `import`ed
normally). This also pulls in the layout strategies used to build the training set.

In [1]:
import os, sys, glob, json, math
import numpy as np, torch
torch.set_num_threads(2)

_HERE = os.path.abspath("../")
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

# Pipeline scripts live in scripts/; it must be on sys.path for their `_pathfix` import.
V6 = os.path.join(_HERE, "scripts")
if V6 not in sys.path:
    sys.path.insert(0, V6)

import importlib.util
spec = importlib.util.spec_from_file_location("de", os.path.join(V6, "04_optimize_differential_evolution_pop.py"))
de = importlib.util.module_from_spec(spec); spec.loader.exec_module(de)
import modules.layouts.strategies as DS
from modules.surrogates import build_recon_from_ckpt

print("loaded module; DEVICE =", de.DEVICE, " N_DETECTORS =", de.N_DETECTORS)
print("run under test:", de.RECON_FOLDER)

loaded module; DEVICE = cpu  N_DETECTORS = 100
run under test: /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/12_10pct_mean_analytic/test_v6_run_03_recentered


## Load the frozen models under test + mountain + primaries
These are exactly the artifacts the optimizers see: the multi-species DeepSets surrogate
(`fnn_<species>.pt`), the DeepSets reconstruction net (`recon.pt`), and the mountain geometry, all
from `modules.constants.RUN_LOCATION`.
`U_layout(E, N)` is the composite utility that the layout optimizers maximize, scored on the
surrogate's **mean** prediction (`sample=False`) so every test below is deterministic — a sampled
draw would make identical layouts score differently and hide real (in)variance behind noise.

In [2]:
fnn, _ = de.load_models(de.DEVICE)          # reuse the dual-species surrogate; ignore the MLP recon

# DeepSets reconstruction net written by 03_train_recon_deepsets.py (= RECON_FOLDER + "_deepsets").
# build_recon_from_ckpt takes the LOADED ckpt dict (not a path), n_det, and device.
OUTPUT_FOLDER = de.RECON_FOLDER + "_deepsets"
recon_ckpt = torch.load(os.path.join(OUTPUT_FOLDER, "recon.pt"), map_location=de.DEVICE)
_recon_ds  = build_recon_from_ckpt(recon_ckpt, de.N_DETECTORS, de.DEVICE)
print(f"[load] recon.pt  model={recon_ckpt['config'].get('model_type')}  "
      f"val={recon_ckpt.get('val_total','?')}  <- {OUTPUT_FOLDER}")

# de.utility_of_xy feeds recon a per-detector (B, n_det, 4) tensor; DeepSetsRecon.forward
# expects exactly that. The adapter still also accepts a flat (B, n_det*4) input (the old
# MLP contract) so every test cell below works unchanged.
class _ReconFlatAdapter(torch.nn.Module):
    def __init__(self, m, n_det, feat=4):
        super().__init__(); self.m, self.n_det, self.feat = m, n_det, feat
    def forward(self, x):
        if x.dim() == 2:
            x = x.reshape(x.shape[0], self.n_det, self.feat)
        return self.m(x)
recon = _ReconFlatAdapter(_recon_ds, de.N_DETECTORS, 4)

mtn = de.load_tr_mountain(de.GEOMETRY_PATH_RESOLVED, de.GEOMETRY_GROUP, de.DET_KEY,
    east_entry=de.EAST_ENTRY, layer_east_dx=de.LAYER_EAST_DX, n_planes=de.N_PLANES)
prim = torch.load(os.path.join(de.TRAINING_DATASET_FOLDER, "primary.pt")).float()
Ntot = prim.shape[0]

# One fixed 2000-shower batch. U is batch-size independent (see Test 1), so 2000 is plenty.
pb = prim[torch.randperm(Ntot, generator=torch.Generator().manual_seed(42))[:2000]]

# Every layout here is (East, North): sample_initial_layout_ne, project_to_mountain_ne and
# all modules.layouts.strategies generators return that order.
def project(E, N):
    return de.project_to_mountain_ne(mtn, torch.as_tensor(E).float(), torch.as_tensor(N).float())

def U_layout(E, N, batch=None, do_project=True):
    """Composite utility U of an (East, North) layout against a primary batch."""
    if do_project:
        E, N = project(E, N)
    # sample=False: the surrogate's mean. The default draws a fresh stochastic sample per call,
    # so the permutation tests would see noise instead of the recon's actual invariance.
    u, r, _ = de.utility_of_xy(torch.as_tensor(E).float(), torch.as_tensor(N).float(),
                               pb if batch is None else batch, fnn, recon, sample=False)
    return float(u)

def clean(scheme):
    E, N = de.sample_initial_layout_ne(mtn, n_units=de.N_DETECTORS, scheme=scheme)
    return project(E, N)

print("models + mountain loaded; primaries:", Ntot)
print("mountain East bbox [%.0f, %.0f]  Up range [%.0f, %.0f]" % (
    mtn.east_lo, mtn.east_hi, mtn.vertices_ENU[:,2].min(), mtn.vertices_ENU[:,2].max()))

[load] fnn_electron.pt  model=deepsets  epoch=101  val=0.06265656416740731
       /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/12_10pct_mean_analytic/test_v6_run_02_recentered/fnn_electron.pt  mtime=2026-09-10 02:22
[load] fnn_muon.pt  model=deepsets  epoch=101  val=0.12209188668429302
       /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/12_10pct_mean_analytic/test_v6_run_02_recentered/fnn_muon.pt  mtime=2026-09-10 02:52
[load] fnn_photon.pt  model=deepsets  epoch=100  val=0.07992766209968287
       /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/12_10pct_mean_analytic/test_v6_run_02_recentered/fnn_photon.pt  mtime=2026-09-10 03:22
[load] recon.pt  model=deepsets  epoch=601  val=0.04370751533578564  <- /n/holylfs05/LABS/arguelles_delgado_lab/Everyone/zdimitrov/detector_optimization_v6/12_10pct_mean_analytic/test_v6_run_03_recentered_deepsets
[load] recon.pt  model=deepsets  

## Test 1 — U is a per-event mean, so it does NOT depend on batch size
`U_E` / `U_angle` average over events, so the objective's *level* is independent of how many
showers you score. This rules out the 50k-vs-512 fixed-batch difference (between
`04_optimize_differential_evolution_pop.py` and the others) as a cause of any U-level difference —
only the **layout** matters.

In [3]:
gE, gN = clean("grid")
for B in (256, 512, 2000):
    b = prim[torch.randperm(Ntot, generator=torch.Generator())[:B]]
    print(f"B={B:6d}  U={U_layout(gE, gN, batch=b, do_project=False):+.2f}")

B=   256  U=+62.14
B=   512  U=+62.04
B=  2000  U=+61.91


## Test 2 — the clean grid scores ~136; center / perturbed score ~25–34
Among hand-built layouts the grid is dramatically higher than a center
cluster or a 1000 m-perturbed grid. (`r_mean` saturates at 1 because `RECONSTRUCT_THRESHOLD=10`
with 100 detectors, so U is driven by angular/energy reconstruction accuracy, not detector count.)

In [4]:
gE, gN = clean("grid"); cE, cN = clean("center")
gen = torch.Generator()
pE, pN = project(gE + torch.randn(100, generator=gen)*1000, gN + torch.randn(100, generator=gen)*1000)
for nm, (E, N) in [("clean grid", (gE, gN)), ("clean center", (cE, cN)),
                   ("perturbed grid s=1000", (pE, pN))]:
    print(f"{nm:22s} U={U_layout(E, N, do_project=False):+.2f}")

clean grid             U=+61.78
clean center           U=+9.69
perturbed grid s=1000  U=+57.32


## Test 3 — peak shape: how fast U falls as the grid is jittered
Add i.i.d. Gaussian noise of std σ (metres) to the grid. U decays smoothly but steeply — a ~200 m
jitter (≈4% of the array's ~4600 m span) already roughly halves it. A genuine physics optimum
would not be this fragile; this sharpness is the first hint of a surrogate artifact.

In [5]:
gE, gN = clean("grid")
for s in (0, 25, 50, 100, 200, 400, 800):
    if s == 0:
        print(f"sigma={s:4d}  U={U_layout(gE, gN, do_project=False):+.2f}"); continue
    us = []
    for seed in range(3):
        g = torch.Generator().manual_seed(seed)
        us.append(U_layout(gE + torch.randn(100, generator=g)*s, gN + torch.randn(100, generator=g)*s))
    print(f"sigma={s:4d}  U={np.mean(us):+.2f} (+/-{np.std(us):.1f})")

sigma=   0  U=+61.78
sigma=  25  U=+61.76 (+/-0.0)
sigma=  50  U=+61.52 (+/-0.2)
sigma= 100  U=+61.33 (+/-0.2)
sigma= 200  U=+61.48 (+/-0.3)
sigma= 400  U=+60.69 (+/-0.5)
sigma= 800  U=+58.18 (+/-0.3)


## Test 4 — U of the ACTUAL training layout strategies
Scores each family in `modules.layouts.strategies.ACTIVE_STRATEGIES` — the layouts the
surrogate/recon were trained on. Currently that is `center_gauss400`, `uniform_random_a/b` and
`latin_hypercube_a/b`: **no grid and no rings**. The clean grid printed last is therefore
out-of-distribution for this checkpoint, so compare its score against these in-distribution
families with that in mind.

In [6]:
for name, fn_name, kw in DS._STRATEGIES:
    fn = DS._STRATEGY_FNS[fn_name]
    us = [U_layout(*fn(mtn, n_det=de.N_DETECTORS, rng=np.random.default_rng(s), **kw), do_project=False)
          for s in range(4)]
    print(f"{name:16s} U={np.mean(us):+7.2f} (+/-{np.std(us):4.1f})   {kw}")
print(f"\nclean grid (no jitter): U={U_layout(*clean('grid'), do_project=False):+.2f}")

center_gauss400  U= +58.79 (+/- 1.7)   {'sigma': 400.0}
uniform_random_a U= +61.56 (+/- 0.4)   {}
uniform_random_b U= +61.56 (+/- 0.4)   {}
latin_hypercube_a U= +61.84 (+/- 0.3)   {}
latin_hypercube_b U= +61.84 (+/- 0.3)   {}

clean grid (no jitter): U=+61.78


## Test 6 — Permute the detector ORDER, keep positions identical
Keep the exact same 100 grid positions; only reorder which detector is fed into which slot of
`recon`. A physically correct (permutation-invariant) reconstructor MUST return the same U —
relabeling detectors can't change the physics. The DeepSets recon pools over detectors, so every
permuted order should print the same U as the canonical one.

In [7]:
gE, gN = clean("grid")
print(f"grid, CANONICAL order:  U={U_layout(gE, gN, do_project=False):+.2f}")
for s in range(3):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    print(f"  permuted order seed{s}: U={U_layout(gE[p], gN[p], do_project=False):+.2f}")

grid, CANONICAL order:  U=+61.78
  permuted order seed0: U=+61.78
  permuted order seed1: U=+61.78
  permuted order seed2: U=+61.78


## Test 7 — it's NOT coverage: rigid shift + naive lattice
A rigid shift of the whole grid (regularity and coverage preserved) decays U smoothly; and a naive
10×10 lattice with good coverage (91/100 distinct positions after mountain projection) still scores
~22. So poor coverage / clumping is not why the other layouts are low.

In [8]:
gE, gN = clean("grid")
for d in (100, 200, 500, 1000):
    print(f"rigid shift +{d:4d} m North: U={U_layout(gE, gN + d):+.2f}")

ax = torch.linspace(float(gE.min()), float(gE.max()), 10)
ay = torch.linspace(float(gN.min()), float(gN.max()), 10)
GX, GY = torch.meshgrid(ax, ay, indexing="ij")
LE, LN = GX.reshape(-1), GY.reshape(-1)
LEp, LNp = project(LE, LN)
uniq = torch.unique(torch.stack([LEp, LNp], -1).round(), dim=0).shape[0]
print(f"naive 10x10 lattice: U={U_layout(LEp, LNp, do_project=False):+.2f}   "
      f"unique positions after projection = {uniq}/100")

rigid shift + 100 m North: U=+60.40
rigid shift + 200 m North: U=+56.80
rigid shift + 500 m North: U=+35.79
rigid shift +1000 m North: U=+11.63
naive 10x10 lattice: U=+62.26   unique positions after projection = 88/100


## Test 8 — measure `recon`'s permutation-(non)invariance directly
Permuting the detector rows changes recon's raw output by **0%** of its
magnitude — so the deepsets produces an invariant function.

In [9]:
def feats(E, N):
    E, N = project(E, N)
    xy = torch.stack([E, N], -1).unsqueeze(0).expand(pb.shape[0], -1, -1)
    pr = fnn(pb, xy)
    return torch.stack([xy[..., 0], xy[..., 1], pr[..., 0], pr[..., 1]], -1)   # (B, 100, 4)

def invariance(name, E, N):
    f = feats(torch.as_tensor(E).float(), torch.as_tensor(N).float())
    o1 = recon(f.reshape(f.shape[0], -1))
    ds = []
    for s in range(5):
        p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
        o2 = recon(f[:, p, :].reshape(f.shape[0], -1))
        ds.append((o1 - o2).abs().mean().item())
    print(f"{name:12s} mean|delta out|={np.mean(ds):.4f}  (|out|~{o1.abs().mean().item():.3f})  "
          f"rel={np.mean(ds)/o1.abs().mean().item():.3f}")

invariance("grid",       *de.sample_initial_layout_ne(mtn, 100, "grid"))
invariance("rings_R800", *DS.layout_rings(mtn, n_det=100, rng=np.random.default_rng(0),
                                          outer_radius=800.0, n_rings=6, jitter_sigma=200.0))
invariance("center",     *DS.layout_center_gaussian(mtn, n_det=100, rng=np.random.default_rng(0),
                                                     sigma=200.0))

grid         mean|delta out|=0.0000  (|out|~0.464)  rel=0.000
rings_R800   mean|delta out|=0.0000  (|out|~0.463)  rel=0.000
center       mean|delta out|=0.0000  (|out|~0.463)  rel=0.000


## Test 9 — the fair (order-averaged) value of the grid
Physics is order-invariant, so the honest score of a layout is its average over detector orderings.

In [10]:
gE, gN = clean("grid")
canonical = U_layout(gE, gN, do_project=False)
print(f"grid, CANONICAL order:  U={canonical:+.2f}")
us = []
for s in range(15):
    p = torch.randperm(100, generator=torch.Generator().manual_seed(s))
    us.append(U_layout(gE[p], gN[p], do_project=False))
us = np.array(us)
print(f"grid, 15 RANDOM orders: mean={us.mean():+.2f}  std={us.std():.2f}  "
      f"min={us.min():+.2f}  max={us.max():+.2f}")
if us.std() > 0:
    print(f"-> canonical is {(canonical - us.mean())/us.std():.1f} sigma from the order-average")
else:
    print("-> every order scores identically: U is exactly permutation-invariant")

grid, CANONICAL order:  U=+61.78
grid, 15 RANDOM orders: mean=+61.78  std=0.00  min=+61.78  max=+61.78
-> canonical is 1.1 sigma from the order-average


## Summary
- Permutation invariariance works 
- the NN is good at reconstruction of all permutaiotons
- shifting the detectors reduces the utility score, which means there is still an issue with the training data